<a href="https://colab.research.google.com/github/DanishShah619/git_agent/blob/main/git_pages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install requests beautifulsoup4 tqdm

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Define the output directory in Google Drive
output_dir = '/content/drive/MyDrive/rag_git/git_scraper_results'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Google Drive mounted. Output directory created at: {output_dir}")

Mounted at /content/drive
Google Drive mounted. Output directory created at: /content/drive/MyDrive/rag_git/git_scraper_results


The Google Drive is mounted, and the output directory is ready. Now I will modify the `scrape_all` function to generate the Markdown file.

In [6]:
import re
import json
import time
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
from html.parser import HTMLParser

# Updated INPUT_FILE to point to Google Drive
# INPUT_FILE = Path("/mnt/user-data/uploads/git_commands_documentation_full.json")
INPUT_FILE  = Path("/content/drive/MyDrive/rag_git/git_scraper_results/git_commands_documentation_full.json")
CACHE_FILE  = Path("/content/drive/MyDrive/rag_git/git_scraper_results/git_docs_cache.json") # Updated cache file path
# OUTPUT_FILE = Path("/mnt/user-data/outputs/git_command_chunks.json")
OUTPUT_FILE = Path("/content/drive/MyDrive/rag_git/git_scraper_results/git_command_chunks.json")

# ── Category mapping ──────────────────────────────────────────────────────────
COMMAND_CATEGORIES = {
    "init": "setup", "clone": "setup", "config": "setup",
    "add": "staging", "mv": "staging", "rm": "staging", "restore": "staging",
    "commit": "history", "tag": "history", "notes": "history",
    "branch": "branching", "checkout": "branching", "switch": "branching",
    "merge": "branching", "rebase": "branching", "cherry-pick": "branching",
    "diff": "inspection", "log": "inspection", "show": "inspection",
    "status": "inspection", "blame": "inspection", "bisect": "inspection",
    "grep": "inspection", "describe": "inspection",
    "stash": "saving", "worktree": "saving",
    "reset": "undoing", "revert": "undoing", "clean": "undoing",
    "fetch": "remote", "pull": "remote", "push": "remote",
    "remote": "remote", "submodule": "remote",
    "reflog": "recovery", "fsck": "recovery", "gc": "maintenance",
    "archive": "maintenance", "bundle": "maintenance",
    "apply": "patching", "am": "patching", "format-patch": "patching",
    "shortlog": "reporting",
}

DIFFICULTY_MAP = {
    "setup": "beginner", "staging": "beginner", "history": "beginner",
    "inspection": "beginner", "branching": "intermediate", "saving": "intermediate",
    "undoing": "intermediate", "remote": "intermediate",
    "patching": "advanced", "recovery": "advanced", "maintenance": "advanced",
    "reporting": "intermediate",
}

# ── Simple HTML → text parser ─────────────────────────────────────────────────
class GitDocParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text_parts = []
        self._skip = False
        self._in_code = False
        self._skip_tags = {"script", "style", "nav", "header", "footer"}
        self._current_tag = ""

    def handle_starttag(self, tag, attrs):
        self._current_tag = tag
        if tag in self._skip_tags:
            self._skip = True
        if tag in ("code", "pre"):
            self._in_code = True
            self.text_parts.append("\n```\n")

    def handle_endtag(self, tag):
        if tag in self._skip_tags:
            self._skip = False
        if tag in ("code", "pre"):
            self._in_code = False
            self.text_parts.append("\n```\n")
        if tag in ("p", "h1", "h2", "h3", "h4", "li", "dt", "dd"):
            self.text_parts.append("\n")

    def handle_data(self, data):
        if not self._skip:
            self.text_parts.append(data)

    def get_text(self):
        return "".join(self.text_parts)


def fetch_url(url: str, timeout: int = 10) -> str | None:
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except Exception as e:
        print(f"    ✗ fetch error: {e}")
        return None


def parse_doc_page(html: str) -> dict:
    """Extract structured content from a git-scm.com/docs page."""
    parser = GitDocParser()
    parser.feed(html)
    raw = parser.get_text()

    # Clean up whitespace
    lines = [l.strip() for l in raw.split("\n")]
    lines = [l for l in lines if l]

    result = {
        "synopsis": "",
        "description": "",
        "options": [],
        "examples": [],
    }

    # Find synopsis block (usually first code block)
    in_synopsis = False
    in_options = False
    synopsis_lines = []
    desc_lines = []
    current_option = None
    option_desc_lines = []

    for i, line in enumerate(lines):
        lower = line.lower()

        if "synopsis" in lower and len(line) < 30:
            in_synopsis = True
            continue
        if in_synopsis:
            if line.startswith("```"):
                if synopsis_lines:  # closing fence
                    in_synopsis = False
                continue
            synopsis_lines.append(line)
            continue

        if any(x in lower for x in ["options", "flags", "configuration"]) and len(line) < 30:
            in_options = True
            continue

        if in_options:
            # Option lines typically start with - or --
            if re.match(r"^-{1,2}[a-zA-Z]", line):
                if current_option and option_desc_lines:
                    result["options"].append({
                        "flag": current_option,
                        "description": " ".join(option_desc_lines).strip()
                    })
                current_option = line
                option_desc_lines = []
            elif current_option and line and not line.startswith("```"):
                option_desc_lines.append(line)
            continue

        # Description: substantial lines before options
        if not in_options and len(line) > 40 and not line.startswith("```"):
            desc_lines.append(line)

    # Flush last option
    if current_option and option_desc_lines:
        result["options"].append({
            "flag": current_option,
            "description": " ".join(option_desc_lines).strip()
        })

    result["synopsis"] = "\n".join(synopsis_lines[:10]).strip()
    result["description"] = " ".join(desc_lines[:6]).strip()[:1200]
    result["options"] = result["options"][:20]  # cap at 20 flags

    return result


def load_cache() -> dict:
    if CACHE_FILE.exists():
        with open(CACHE_FILE) as f:
            return json.load(f)
    return {}


def save_cache(cache: dict):
    # Ensure the cache directory exists before trying to write
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)


def build_chunk_text(cmd: str, short_desc: str, doc: dict) -> str:
    """Compose the full text of a command chunk."""
    parts = [f"# git {cmd}\n"]
    parts.append(f"{short_desc}\n")

    if doc["synopsis"]:
        parts.append(f"\n## Synopsis\n```\n{doc['synopsis']}\n```")

    if doc["description"]:
        parts.append(f"\n## Description\n{doc['description']}")

    if doc["options"]:
        parts.append("\n## Key Options")
        for opt in doc["options"][:12]:
            flag = opt["flag"]
            desc = opt["description"][:200]
            parts.append(f"- `{flag}`: {desc}")

    return "\n".join(parts)


def make_chunk_id(cmd: str) -> str:
    return "cmd_" + hashlib.md5(cmd.encode()).hexdigest()[:10]


def get_related_commands(cmd: str, category: str, all_commands: list) -> list:
    """Return other commands in the same category."""
    same_cat = [c for c in all_commands
                if COMMAND_CATEGORIES.get(c, "") == category and c != cmd]
    return same_cat[:5]


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Ensure the output directory for INPUT_FILE exists
    INPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

    with open(INPUT_FILE) as f:
        raw_data = json.load(f)

    # Filter to real git commands (skip meta entries like "Credential helpers")
    commands = [
        item for item in raw_data
        if re.match(r"^[a-z]", item["command"]) and "git-scm.com/docs/git" in item["url"]
    ]
    all_cmd_names = [item["command"] for item in commands]

    cache = load_cache()
    chunks = []

    print(f"Processing {len(commands)} commands...\n")

    for item in commands:
        cmd = item["command"]
        url = item["url"]
        short_desc = item["full_details"] or f"git-{cmd}"

        print(f"  [{cmd}]", end=" ", flush=True)

        # Try cache first
        if url in cache:
            doc = cache[url]
            print("(cached)")
        else:
            html = fetch_url(url)
            if html:
                doc = parse_doc_page(html)
                cache[url] = doc
                save_cache(cache)
                print(f"✓ ({len(doc['options'])} flags)")
            else:
                doc = {"synopsis": "", "description": "", "options": [], "examples": []}
                print("✗ (using stub)")
            time.sleep(0.4)  # be polite

        category = COMMAND_CATEGORIES.get(cmd, "general")
        difficulty = DIFFICULTY_MAP.get(category, "intermediate")
        related = get_related_commands(cmd, category, all_cmd_names)
        text = build_chunk_text(cmd, short_desc, doc)

        chunks.append({
            "chunk_id": make_chunk_id(cmd),
            "chunk_type": "command_reference",
            "source": "git-scm-docs",
            "source_url": url,
            "command": f"git {cmd}",
            "category": category,
            "difficulty": difficulty,
            "related_commands": related,
            "synopsis": doc["synopsis"],
            "flags": doc["options"],
            "text": text,
            "token_estimate": len(text.split()),
        })

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_FILE, "w") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    total_tokens = sum(c["token_estimate"] for c in chunks)
    print(f"\nDone. {len(chunks)} command chunks | ~{total_tokens:,} tokens")
    print(f"Output: {OUTPUT_FILE}")

Processing 85 commands...

  [git] ✓ (20 flags)
  [config] ✓ (20 flags)
  [help] ✓ (15 flags)
  [bugreport] ✓ (5 flags)
  [init] ✓ (13 flags)
  [clone] ✓ (20 flags)
  [add] ✓ (20 flags)
  [status] ✓ (20 flags)
  [diff] ✓ (20 flags)
  [commit] ✓ (20 flags)
  [notes] ✓ (20 flags)
  [restore] ✓ (20 flags)
  [reset] ✓ (20 flags)
  [rm] ✓ (19 flags)
  [mv] ✓ (6 flags)
  [branch] ✓ (20 flags)
  [checkout] ✓ (20 flags)
  [switch] ✓ (20 flags)
  [merge] ✓ (20 flags)
  [mergetool] ✓ (20 flags)
  [log] ✓ (20 flags)
  [stash] ✓ (20 flags)
  [tag] ✓ (20 flags)
  [worktree] ✓ (20 flags)
  [fetch] ✓ (20 flags)
  [pull] ✓ (20 flags)
  [push] ✓ (20 flags)
  [remote] ✓ (20 flags)
  [submodule] ✓ (20 flags)
  [show] ✓ (20 flags)
  [log] (cached)
  [diff] (cached)
  [difftool] ✓ (20 flags)
  [range-diff] ✓ (17 flags)
  [shortlog] ✓ (20 flags)
  [describe] ✓ (18 flags)
  [apply] ✓ (20 flags)
  [cherry-pick] ✓ (20 flags)
  [diff] (cached)
  [rebase] ✓ (20 flags)
  [revert] ✓ (20 flags)
  [bisect] ✓ (4 flag